### Download de date pentru inserat in tabele

In [ ]:
%pip install yfinance
%pip install matplotlib
%pip install pandas
%pip install numpy

In [ ]:
# import modules
from datetime import datetime
import yfinance as yf
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
# Metode pentru extras date de la bursa
class StockPrice:
   ticker: str
   id_bursa: str | None
   data_cotatie: str # of the form YYYY-MM-DD HH:MM:SS
   pret_deschidere: np.float64
   pret_inchidere: np.float64
   pret_maxim: np.float64
   pret_minim: np.float64
   volum: np.float64
   
   def __init__(self, ticker: str, id_bursa: str | None, data_cotatie: str, pret_deschidere: np.float64, pret_inchidere: np.float64, pret_maxim: np.float64, pret_minim: np.float64, volum: np.float64):
      # save ticker without exchange suffix (e.g. TLV.RO -> TLV)
      self.ticker = ticker.rsplit('.', 1)[0]
      self.id_bursa = id_bursa
      self.data_cotatie = data_cotatie
      self.pret_deschidere = pret_deschidere
      self.pret_inchidere = pret_inchidere
      self.pret_maxim = pret_maxim
      self.pret_minim = pret_minim
      self.volum = volum

   def __str__(self):
      return f"StockPrice(ticker={self.ticker}, id_bursa={self.id_bursa}, data_cotatie={self.data_cotatie}, pret_deschidere={self.pret_deschidere}, pret_inchidere={self.pret_inchidere}, pret_maxim={self.pret_maxim}, pret_minim={self.pret_minim}, volum={self.volum})"


def conv_yf_to_stock_price(yf_data: pd.DataFrame, ticker: str, id_bursa: str | None) -> list[StockPrice]:
   """
   Input va fi de forma: MultiIndex([
      ( 'Close', 'NVDA'),
      (  'High', 'NVDA'),
      (   'Low', 'NVDA'),
      (  'Open', 'NVDA'),
      ('Volume', 'NVDA')],
      names=['Price', 'Ticker'])
   Coloana de index va fi timestamp-ul
   
   Returneaza un array de obiecte de tip StockPrice, cate unul pentru fiecare rand din yf_data
   """
   
   stock_prices = []
   for index, row in yf_data.iterrows():
      # type of index is actually pd.Timestamp
      idx_timestamp: pd.Timestamp = index # type: ignore 
      data_cotatie = idx_timestamp.strftime('%Y-%m-%d %H:%M:%S')
      
      pret_deschidere = row['Open'].values[0]
      if pret_deschidere is None:
         raise ValueError(f"Open price is None for date {data_cotatie}")
      
      pret_inchidere = row['Close'].values[0]
      if pret_inchidere is None:
         raise ValueError(f"Close price is None for date {data_cotatie}")
      
      pret_maxim = row['High'].values[0]
      if pret_maxim is None:
         raise ValueError(f"High price is None for date {data_cotatie}")
      
      pret_minim = row['Low'].values[0]
      if pret_minim is None:
         raise ValueError(f"Low price is None for date {data_cotatie}")
      
      volum = row['Volume'].values[0]
      if volum is None:
         raise ValueError(f"Volume is None for date {data_cotatie}")
      
      stock_price = StockPrice(ticker, id_bursa, data_cotatie, pret_deschidere, pret_inchidere, pret_maxim, pret_minim, volum)
      stock_prices.append(stock_price)
   return stock_prices


def download_ticker(ticker: str, id_bursa: str | None, start_date: str, end_date: str, interval: str = '1d', show_graph: bool = False) -> list[StockPrice]:
   """
   Download the stock price data for the given ticker and date range using yfinance.
   id_bursa is explicitly provided (e.g. NYSE, NASDAQ, MEXI, AEB, BVB).
   Return a list of StockPrice objects.
   """
   yf_data: pd.DataFrame | None = yf.download(
      tickers=ticker, 
      start=start_date, 
      end=end_date, 
      interval=interval)
   if yf_data is None:
      raise ValueError(f"Failed to download data for ticker {ticker}")
   
   if show_graph:
      plt.figure(figsize=(10, 5))
      plt.plot(yf_data.index, yf_data['Close'])
      plt.title(f'{ticker} Stock Price')
      plt.xlabel('Date')
      plt.ylabel('Close Price')
      plt.grid()
      plt.show()
   
   return conv_yf_to_stock_price(yf_data, ticker, id_bursa)


In [ ]:
# # Ex de utilizare
# stock_prices = download_ticker(
#    ticker='TLV.RO',
#    id_bursa='BVB',
#    start_date='2025-01-01',
#    end_date=datetime.now().strftime('%Y-%m-%d'),
#    interval='1wk', show_graph=True)
# len(stock_prices[:25])


In [ ]:
target_companies = [
   ('TLV.RO', 'BVB'), # Banca transilvania
   ('NVDA', 'NASDAQ'), # NVIDIA
   ('ALV.DE', 'FSE'), # Allianz Group
   ('NVO', 'NYSE'), # Novo Nordisk
   ('AAPL', 'NASDAQ') # Apple
]

stock_prices = download_ticker(ticker='ALV.DE', id_bursa='FSE', 
   start_date='2025-06-01', end_date=datetime.now().strftime('%Y-%m-%d'), 
   interval='1wk', show_graph=True)
len(stock_prices)